In [1]:
import pandas as pd

nav = pd.read_csv("../data/raw/02_nav_history.csv")
print(nav.shape)
nav.head()

(46000, 3)


,amfi_code,date,nav
0,119551,2022-01-03,54.3856
1,119551,2022-01-04,54.3474
2,119551,2022-01-05,54.6869
3,119551,2022-01-06,55.4550
4,119551,2022-01-07,55.3692


In [2]:
# Parse dates
nav["date"] = pd.to_datetime(nav["date"])

# Sort by amfi_code + date
nav = nav.sort_values(["amfi_code", "date"]).reset_index(drop=True)

# Remove duplicates
before = len(nav)
nav = nav.drop_duplicates(subset=["amfi_code", "date"])
print(f"Removed {before - len(nav)} duplicate rows")

Removed 0 duplicate rows


In [3]:
filled_groups = []

for code, group in nav.groupby("amfi_code"):
    full_range = pd.date_range(group["date"].min(), group["date"].max(), freq="D")
    group = group.set_index("date").reindex(full_range)
    group["nav"] = group["nav"].ffill()
    group["amfi_code"] = code   # just assign it directly, no need to ffill
    group.index.name = "date"
    filled_groups.append(group.reset_index())

nav = pd.concat(filled_groups, ignore_index=True)
print(nav.shape)
nav.head()

(64320, 3)


,date,amfi_code,nav
0,2022-01-03,100016,520.4608
1,2022-01-04,100016,515.0971
2,2022-01-05,100016,521.7239
3,2022-01-06,100016,515.7880
4,2022-01-07,100016,515.1639


In [4]:
invalid = nav[nav["nav"] <= 0]
print(f"Invalid NAV rows (<=0): {len(invalid)}")
nav = nav[nav["nav"] > 0]

Invalid NAV rows (<=0): 0


In [5]:
nav.to_csv("../data/processed/nav_history_clean.csv", index=False)
print("Saved!")

Saved!


In [6]:
txn = pd.read_csv("../data/raw/08_investor_transactions.csv")
print(txn.shape)
print(txn["transaction_type"].unique())
print(txn["kyc_status"].unique())
txn.head()

(32778, 13)
<StringArray>
['SIP', 'Redemption', 'Lumpsum']
Length: 3, dtype: str
<StringArray>
['Verified', 'Pending']
Length: 2, dtype: str


,investor_id,transaction_date,amfi_code,transaction_type,amount_inr,state,city,city_tier,age_group,gender,annual_income_lakh,payment_mode,kyc_status
0,INV003054,2024-01-01,119092,SIP,1834,Telangana,Hyderabad,T30,56+,Female,77.1,UPI,Verified
1,INV002952,2024-01-01,148567,Redemption,392882,Punjab,Amritsar,B30,18-25,Male,7.1,Cheque,Verified
2,INV003420,2024-01-01,118636,SIP,912,Haryana,Faridabad,B30,36-45,Male,47.2,Mandate,Verified
3,INV003436,2024-01-01,118634,SIP,1102,Maharashtra,Mumbai,T30,36-45,Female,54.4,Cheque,Pending
4,INV004691,2024-01-01,119094,Lumpsum,8682,Delhi,Noida,T30,26-35,Male,14.5,Net Banking,Pending


In [7]:
# Standardise (defensive - already clean, but ensures consistency)
txn["transaction_type"] = txn["transaction_type"].str.strip().str.title()
txn["kyc_status"] = txn["kyc_status"].str.strip().str.title()

# Validate amount > 0
invalid_amt = txn[txn["amount_inr"] <= 0]
print(f"Invalid amounts (<=0): {len(invalid_amt)}")
txn = txn[txn["amount_inr"] > 0]

# Fix date format
txn["transaction_date"] = pd.to_datetime(txn["transaction_date"], errors="coerce")
print(f"Unparseable dates: {txn['transaction_date'].isna().sum()}")

# Check for any nulls in kyc_status enum
print(f"Null KYC status: {txn['kyc_status'].isna().sum()}")

print(txn.shape)
txn.head()

Invalid amounts (<=0): 0
Unparseable dates: 0
Null KYC status: 0
(32778, 13)


,investor_id,transaction_date,amfi_code,transaction_type,amount_inr,state,city,city_tier,age_group,gender,annual_income_lakh,payment_mode,kyc_status
0,INV003054,2024-01-01,119092,Sip,1834,Telangana,Hyderabad,T30,56+,Female,77.1,UPI,Verified
1,INV002952,2024-01-01,148567,Redemption,392882,Punjab,Amritsar,B30,18-25,Male,7.1,Cheque,Verified
2,INV003420,2024-01-01,118636,Sip,912,Haryana,Faridabad,B30,36-45,Male,47.2,Mandate,Verified
3,INV003436,2024-01-01,118634,Sip,1102,Maharashtra,Mumbai,T30,36-45,Female,54.4,Cheque,Pending
4,INV004691,2024-01-01,119094,Lumpsum,8682,Delhi,Noida,T30,26-35,Male,14.5,Net Banking,Pending


In [8]:
txn.to_csv("../data/processed/investor_transactions_clean.csv", index=False)
print("Saved!")

Saved!


In [9]:
perf = pd.read_csv("../data/raw/07_scheme_performance.csv")
print(perf.shape)
perf.head()

(40, 19)


,amfi_code,scheme_name,fund_house,category,plan,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,beta,sharpe_ratio,sortino_ratio,std_dev_ann_pct,max_drawdown_pct,aum_crore,expense_ratio_pct,morningstar_rating,risk_grade
0,119551,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund,Large Cap,Regular,12.42,12.36,14.45,11.49,0.87,0.89,0.88,1.29,14.0,-21.70,14288,1.54,4,Moderate
1,119552,SBI Bluechip Fund - Direct Plan - Growth,SBI Mutual Fund,Large Cap,Direct,15.25,11.30,14.23,9.52,1.78,0.87,0.81,1.29,14.0,-24.43,1231,0.66,3,Moderate
2,119598,SBI Small Cap Fund - Regular Plan - Growth,SBI Mutual Fund,Small Cap,Regular,24.56,23.39,20.67,22.16,1.23,0.89,0.94,1.35,25.0,-13.35,19259,1.43,5,Very High
3,119599,SBI Small Cap Fund - Direct Plan - Growth,SBI Mutual Fund,Small Cap,Direct,20.59,23.14,21.82,22.01,1.13,1.04,0.93,1.67,25.0,-24.78,36061,0.72,4,Very High
4,119120,SBI Magnum Gilt Fund - Regular Plan - Growth,SBI Mutual Fund,Gilt,Regular,5.34,6.07,5.43,4.47,1.60,0.22,1.52,2.11,4.0,-2.30,24101,0.77,5,Low


In [10]:
return_cols = ["return_1yr_pct", "return_3yr_pct", "return_5yr_pct", "benchmark_3yr_pct",
               "alpha", "beta", "sharpe_ratio", "sortino_ratio", "std_dev_ann_pct", "max_drawdown_pct"]

# Validate numeric
for col in return_cols:
    perf[col] = pd.to_numeric(perf[col], errors="coerce")
    non_numeric = perf[col].isna().sum()
    if non_numeric > 0:
        print(f"{col}: {non_numeric} non-numeric values")

print("Numeric validation done.")

Numeric validation done.


In [11]:
for col in ["return_1yr_pct", "return_3yr_pct", "return_5yr_pct"]:
    mean, std = perf[col].mean(), perf[col].std()
    flag_col = f"{col}_anomaly"
    perf[flag_col] = (perf[col] - mean).abs() > 3 * std
    print(f"{col} anomalies: {perf[flag_col].sum()}")

return_1yr_pct anomalies: 0
return_3yr_pct anomalies: 0
return_5yr_pct anomalies: 0


In [12]:
out_of_range = perf[(perf["expense_ratio_pct"] < 0.1) | (perf["expense_ratio_pct"] > 2.5)]
print(f"Expense ratio out of range (0.1%-2.5%): {len(out_of_range)}")
perf["expense_ratio_flag"] = (perf["expense_ratio_pct"] < 0.1) | (perf["expense_ratio_pct"] > 2.5)

Expense ratio out of range (0.1%-2.5%): 0


In [13]:
perf.to_csv("../data/processed/scheme_performance_clean.csv", index=False)
print("Saved!")

Saved!


In [14]:
remaining_files = {
    "01_fund_master": None,              # no date column
    "03_aum_by_fund_house": "date",
    "04_monthly_sip_inflows": "month",
    "05_category_inflows": "month",
    "06_industry_folio_count": "month",
    "09_portfolio_holdings": "portfolio_date",
    "10_benchmark_indices": "date",
}

for fname, date_col in remaining_files.items():
    df = pd.read_csv(f"../data/raw/{fname}.csv")
    before = len(df)
    df = df.drop_duplicates()
    if date_col:
        df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
    df.to_csv(f"../data/processed/{fname.split('_',1)[1] if fname[:2].isdigit() else fname}_clean.csv", index=False)
    print(f"{fname}: {before} -> {len(df)} rows, saved.")

01_fund_master: 40 -> 40 rows, saved.
03_aum_by_fund_house: 90 -> 90 rows, saved.
04_monthly_sip_inflows: 48 -> 48 rows, saved.
05_category_inflows: 144 -> 144 rows, saved.
06_industry_folio_count: 21 -> 21 rows, saved.
09_portfolio_holdings: 322 -> 322 rows, saved.
10_benchmark_indices: 8050 -> 8050 rows, saved.


In [15]:
import os
print(os.listdir("../data/processed"))

['.gitkeep', 'aum_by_fund_house_clean.csv', 'benchmark_indices_clean.csv', 'category_inflows_clean.csv', 'fund_master_clean.csv', 'industry_folio_count_clean.csv', 'investor_transactions_clean.csv', 'monthly_sip_inflows_clean.csv', 'nav_history_clean.csv', 'portfolio_holdings_clean.csv', 'scheme_performance_clean.csv']


In [17]:
from sqlalchemy import create_engine

engine = create_engine("sqlite:///../bluestock_mf.db")

# Drop existing tables first (so this cell is safe to re-run)
drop_statements = [
    "DROP TABLE IF EXISTS fact_nav",
    "DROP TABLE IF EXISTS fact_transactions",
    "DROP TABLE IF EXISTS fact_performance",
    "DROP TABLE IF EXISTS fact_aum",
    "DROP TABLE IF EXISTS dim_fund",
    "DROP TABLE IF EXISTS dim_date",
]

with engine.connect() as conn:
    for stmt in drop_statements:
        conn.exec_driver_sql(stmt)
    conn.commit()

# Now create fresh tables
with open("../sql/schema.sql") as f:
    schema_sql = f.read()

with engine.connect() as conn:
    for statement in schema_sql.split(";"):
        if statement.strip():
            conn.exec_driver_sql(statement)
    conn.commit()

print("Schema created!")

Schema created!


In [18]:
import pandas as pd
from sqlalchemy import create_engine

fund_master = pd.read_csv("../data/processed/fund_master_clean.csv")

dim_fund_cols = ["amfi_code", "fund_house", "scheme_name", "category", "sub_category",
                  "plan", "launch_date", "benchmark", "expense_ratio_pct", "exit_load_pct",
                  "fund_manager", "risk_category", "sebi_category_code"]

fund_master[dim_fund_cols].to_sql("dim_fund", engine, if_exists="append", index=False)
print(f"dim_fund loaded: {len(fund_master)} rows")

dim_fund loaded: 40 rows


In [19]:
nav = pd.read_csv("../data/processed/nav_history_clean.csv")
nav["date"] = pd.to_datetime(nav["date"])

dates = pd.DataFrame({"date_id": pd.date_range(nav["date"].min(), nav["date"].max())})
dates["year"] = dates["date_id"].dt.year
dates["month"] = dates["date_id"].dt.month
dates["quarter"] = dates["date_id"].dt.quarter
dates["day_of_week"] = dates["date_id"].dt.day_name()

dates.to_sql("dim_date", engine, if_exists="append", index=False)
print(f"dim_date loaded: {len(dates)} rows")

dim_date loaded: 1608 rows


In [20]:
nav.to_sql("fact_nav", engine, if_exists="append", index=False)
print(f"fact_nav loaded: {len(nav)} rows")

fact_nav loaded: 64320 rows


In [21]:
txn = pd.read_csv("../data/processed/investor_transactions_clean.csv")
txn.to_sql("fact_transactions", engine, if_exists="append", index=False)
print(f"fact_transactions loaded: {len(txn)} rows")

fact_transactions loaded: 32778 rows


In [22]:
perf = pd.read_csv("../data/processed/scheme_performance_clean.csv")
perf_cols = ["amfi_code","return_1yr_pct","return_3yr_pct","return_5yr_pct",
             "sharpe_ratio","max_drawdown_pct","morningstar_rating","risk_grade"]
perf[perf_cols].to_sql("fact_performance", engine, if_exists="append", index=False)
print(f"fact_performance loaded: {len(perf)} rows")

fact_performance loaded: 40 rows


In [23]:
aum = pd.read_csv("../data/processed/aum_by_fund_house_clean.csv")
aum_cols = ["date","fund_house","aum_crore","num_schemes"]
aum[aum_cols].to_sql("fact_aum", engine, if_exists="append", index=False)
print(f"fact_aum loaded: {len(aum)} rows")

fact_aum loaded: 90 rows


In [24]:
tables = {"dim_fund": fund_master, "fact_nav": nav, "fact_transactions": txn,
          "fact_performance": perf, "fact_aum": aum}

for table, df in tables.items():
    count = pd.read_sql(f"SELECT COUNT(*) as n FROM {table}", engine).iloc[0]["n"]
    match = "✅" if len(df) == count else "❌"
    print(f"{table}: source={len(df)}, loaded={count} {match}")

dim_fund: source=40, loaded=40 ✅
fact_nav: source=64320, loaded=64320 ✅
fact_transactions: source=32778, loaded=32778 ✅
fact_performance: source=40, loaded=40 ✅
fact_aum: source=90, loaded=90 ✅


In [25]:
queries = {
    "1_top5_aum": "SELECT fund_house, AVG(aum_crore) as avg_aum FROM fact_aum GROUP BY fund_house ORDER BY avg_aum DESC LIMIT 5",
    "2_avg_nav_month": "SELECT amfi_code, strftime('%Y-%m', date) as month, AVG(nav) as avg_nav FROM fact_nav GROUP BY amfi_code, month",
    "3_sip_yoy": "SELECT strftime('%Y', transaction_date) as year, SUM(amount_inr) as total_sip FROM fact_transactions WHERE transaction_type = 'Sip' GROUP BY year",
    "4_txn_by_state": "SELECT state, COUNT(*) as txn_count, SUM(amount_inr) as total_amount FROM fact_transactions GROUP BY state ORDER BY total_amount DESC",
    "5_low_expense": "SELECT scheme_name, expense_ratio_pct FROM dim_fund WHERE expense_ratio_pct < 1.0",
    "6_top5_return": "SELECT df.scheme_name, fp.return_3yr_pct FROM fact_performance fp JOIN dim_fund df ON fp.amfi_code = df.amfi_code ORDER BY fp.return_3yr_pct DESC LIMIT 5",
    "7_avg_by_age": "SELECT age_group, AVG(amount_inr) as avg_amount FROM fact_transactions GROUP BY age_group",
    "8_fund_by_category": "SELECT category, COUNT(*) as fund_count FROM dim_fund GROUP BY category",
    "9_top_sharpe": "SELECT df.scheme_name, fp.sharpe_ratio FROM fact_performance fp JOIN dim_fund df ON fp.amfi_code = df.amfi_code ORDER BY fp.sharpe_ratio DESC LIMIT 10",
    "10_txn_by_tier": "SELECT city_tier, transaction_type, COUNT(*) as cnt, SUM(amount_inr) as total FROM fact_transactions GROUP BY city_tier, transaction_type",
}

for name, q in queries.items():
    result = pd.read_sql(q, engine)
    print(f"--- {name} ---")
    print(result.head())
    print()

--- 1_top5_aum ---
            fund_house        avg_aum
0      SBI Mutual Fund  943444.444444
1  ICICI Prudential MF  699222.222222
2     HDFC Mutual Fund  636888.888889
3      Nippon India MF  434333.333333
4    Kotak Mahindra MF  389111.111111

--- 2_avg_nav_month ---
  amfi_code    month     avg_nav
0    100016  2022-01  511.923007
1    100016  2022-02  514.538068
2    100016  2022-03  522.286481
3    100016  2022-04  526.114670
4    100016  2022-05  504.335713

--- 3_sip_yoy ---
   year    total_sip
0  2024  153233052.0
1  2025   64000439.0

--- 4_txn_by_state ---
            state  txn_count  total_amount
0          Punjab       2965   315780459.0
1      Tamil Nadu       2806   315177237.0
2  Madhya Pradesh       2931   308312493.0
3       Rajasthan       2577   298645822.0
4         Gujarat       2780   298358940.0

--- 5_low_expense ---
                                         scheme_name  expense_ratio_pct
0           SBI Bluechip Fund - Direct Plan - Growth               0.66